# CLV Model — score the new 100,000 customers\n\nTrains a regressor on the seed 100,000 customers to predict `CLVScore` (0–100), then applies it to the new 100,000 and writes into `FactCustomerCLV`. `CLVBand` is derived from the predicted score with the same thresholds as the warehouse generator (>70 High, >40 Medium, else Low). `CLVSegment` is a direct pass-through of `CustomerType` (Corporate Value / Retail Value / SME Value) — the original generator doesn't model this either, it's a fixed label.\n\nSame caveat as every notebook here: `CLVScore` in the seed data is a formula of `CustomerId`, not real customer value.

In [1]:
from datetime import date

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from db_utils import bulk_insert, get_connection

FEATURE_COLS = [
    "Age", "Gender", "Region", "CustomerType", "Segment", "CustomerStatus",
    "Balance", "AccountType",
]
TODAY = date.today()
CLV_SEGMENT_MAP = {"Corporate": "Corporate Value", "Retail": "Retail Value", "SME": "SME Value"}

conn = get_connection()
seed = pd.read_sql(
    """
    SELECT c.CustomerId, c.Age, c.Gender, c.Region, c.CustomerType, c.Segment, c.CustomerStatus,
           a.Balance, a.AccountType,
           clv.CLVScore
    FROM dbo.DimCustomer c
    JOIN dbo.FactCustomerAccount a ON a.CustomerId = c.CustomerId
    JOIN dbo.FactCustomerCLV clv ON clv.CustomerId = c.CustomerId
    WHERE c.CustomerId <= 100000;
    """,
    conn,
)
conn.close()

new_customers = pd.read_csv("data/new_customers_features.csv")
print("Seed:", seed.shape, " New:", new_customers.shape)

/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_38527/2345601839.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  seed = pd.read_sql(


Seed: (100000, 10)  New: (100000, 15)


In [2]:
combined = pd.concat([seed[FEATURE_COLS], new_customers[FEATURE_COLS]], keys=["seed", "new"])
combined_encoded = pd.get_dummies(combined, drop_first=True)

X_seed = combined_encoded.loc["seed"].reset_index(drop=True)
X_new = combined_encoded.loc["new"].reset_index(drop=True)
y_seed = seed["CLVScore"].reset_index(drop=True)

X_train, X_test, y_train, y_test = train_test_split(X_seed, y_seed, test_size=0.2, random_state=42)
eval_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
eval_model.fit(X_train, y_train)
y_pred = eval_model.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))
print("R2:", round(r2_score(y_test, y_pred), 4))

MAE: 25.6903
R2: 0.1978


In [3]:
final_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
final_model.fit(X_seed, y_seed)

predicted_clv = np.clip(final_model.predict(X_new), 0, 100).round(2)

results = pd.DataFrame({
    "CustomerId": new_customers["CustomerId"],
    "CLVScore": predicted_clv,
    "CLVBand": np.select(
        [predicted_clv > 70, predicted_clv > 40],
        ["High", "Medium"],
        default="Low",
    ),
    "CLVSegment": new_customers["CustomerType"].map(CLV_SEGMENT_MAP),
    "ModelDate": TODAY,
})

print(results["CLVBand"].value_counts())
results.head()

CLVBand
Low       92908
Medium     6973
High        119
Name: count, dtype: int64


,CustomerId,CLVScore,CLVBand,CLVSegment,ModelDate
0,100001,28.53,Low,SME Value,2026-07-23
1,100002,30.59,Low,Retail Value,2026-07-23
2,100003,26.28,Low,SME Value,2026-07-23
3,100004,39.59,Low,Retail Value,2026-07-23
4,100005,44.54,Medium,Retail Value,2026-07-23


In [4]:
cols = ["CustomerId", "CLVScore", "CLVBand", "CLVSegment", "ModelDate"]

conn = get_connection()
n = bulk_insert(conn, "dbo.FactCustomerCLV", cols, list(results[cols].itertuples(index=False, name=None)))
check = pd.read_sql("SELECT COUNT(*) AS NewCLVRows FROM dbo.FactCustomerCLV WHERE CustomerId > 100000;", conn)
conn.close()
print(f"Inserted {n:,} rows into FactCustomerCLV")
check

Inserted 100,000 rows into FactCustomerCLV


/var/folders/xy/sts9z3rj3w3_957f1n6sltrm0000gn/T/ipykernel_38527/4080269090.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql("SELECT COUNT(*) AS NewCLVRows FROM dbo.FactCustomerCLV WHERE CustomerId > 100000;", conn)


,NewCLVRows
0,100000
